In [ ]:
#@title ARL — is every addition worth it? (run this one cell)
# Runtime > Change runtime type > A100 GPU (High-RAM). Results, checkpoints and logs are saved under RESULTS on
# Google Drive; if Colab disconnects, run the cell again and it continues where it stopped.
MODE       = 'quick'   # 'quick': 2 variants x 2 epochs, checks the whole pipeline | 'full': the study
SEEDS      = 3         # 'full' only: training seeds per variant (+1 validation model per variant)
VARIANTS   = 'all'     # 'full' only: 'all' (9 variants) | 'core' (base, A1, A2, A3, A1A2A3) | e.g. 'core avg'
PRECISION  = '32-true' # as TrackTacular; 'bf16-mixed' is faster on an A100
RESULTS    = '/content/drive/MyDrive/retrack_results'
DATA_CACHE = '/content/drive/MyDrive/datasets/wildtrack.tar'  # WILDTRACK copy on Drive (~10 GB); None = always Kaggle
FRESH_EVAL = True      # 'quick' only: redo the evaluations with the current code (finished trainings are reused)
REPO       = 'github.com/rizwanmustafa/retrack.git'
BRANCH     = 'feat/arl-attempt-1'

import os, glob
from google.colab import drive, userdata
drive.mount('/content/drive')

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

# Optional Colab Secrets (key icon on the left): KAGGLE_USERNAME / KAGGLE_KEY if the WILDTRACK download is refused,
# GITHUB_TOKEN if the repository is private.
for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
    if secret(k):
        os.environ[k] = secret(k)

# 1. code: always the latest commit of BRANCH
url = f"https://{secret('GITHUB_TOKEN')}@{REPO}" if secret('GITHUB_TOKEN') else f'https://{REPO}'
if os.path.exists('/content/retrack'):
    !git -C /content/retrack fetch -q origin {BRANCH} && git -C /content/retrack checkout -q {BRANCH} && git -C /content/retrack reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {url} /content/retrack
!git -C /content/retrack log --oneline -1

# 2. data: already in this session -> restore from Drive -> download from Kaggle (only the very first time)
W = '/content/data/Wildtrack'
have = lambda: all(os.path.exists(f'{W}/{d}') for d in ('Image_subsets', 'annotations_positions', 'calibrations'))
if have():
    print('WILDTRACK already in this session')
elif DATA_CACHE and os.path.exists(DATA_CACHE):
    print('restoring WILDTRACK from Drive ...')
    os.makedirs('/content/data', exist_ok=True)
    !tar -xf {DATA_CACHE} -C /content/data
else:
    print('downloading WILDTRACK from Kaggle (first time only) ...')
    !pip -q install kagglehub
    !cd /content/retrack/WorldTrack && python -c "from retrack_pipeline import prepare_wildtrack; prepare_wildtrack('/content/data')"
assert have(), 'WILDTRACK is not ready'
if DATA_CACHE and not os.path.exists(DATA_CACHE):
    print(f'saving WILDTRACK to Drive for later sessions -> {DATA_CACHE}')
    os.makedirs(os.path.dirname(DATA_CACHE), exist_ok=True)
    !tar -chf {DATA_CACHE}.part -C /content/data Wildtrack && mv {DATA_CACHE}.part {DATA_CACHE}

# 3. run (the pipeline sees the data in /content/data and skips its own download)
out = f'{RESULTS}/arl_{MODE}'
if MODE == 'quick' and FRESH_EVAL and os.path.exists(f'{out}/eval'):
    !rm -rf {out}/eval {out}/report {out}/best
extra = f'--precision {PRECISION}'
if MODE == 'full':
    extra += f' --seeds {SEEDS} --variants {VARIANTS}'
!PIPELINE=arl bash /content/retrack/colab/run.sh {MODE} {RESULTS} {extra}

from IPython.display import Markdown, Image, display
if os.path.exists(f'{out}/report/REPORT.md'):
    display(Markdown(open(f'{out}/report/REPORT.md').read()))
    if os.path.exists(f'{out}/report/additions.png'):
        display(Image(f'{out}/report/additions.png', width=720))
print('Everything for this run:', out)
print('Best checkpoints:        ', f'{out}/best')
print('Share for analysis:      ', f'{out}/for_claude.zip  (everything except checkpoints)')
